# Week 15: Graph Neural Networks & Geometric Deep Learning
## Notebook 07: Heterogeneous Graph Neural Networks & RGCN

### 1. Conceptual Intuition & Theoretical Background
Standard Graph Convolutional Networks (GCNs) assume **homogeneous graphs** where all nodes share a single type and all edges represent identical relationships.

Real-world graphs are **heterogeneous** (e.g. Academic Graph: Nodes $\in \{\text{Author}, \text{Paper}, \text{Venue}\}$, Edges $\in \{\text{writes}, \text{cites}, \text{published\_at}\}$).

**Relational Graph Convolutional Networks (RGCN)** assign relation-specific transformation matrices $W_r$:
$$h_i^{(l+1)} = \sigma\left( W_0^{(l)} h_i^{(l)} + \sum_{r \in \mathcal{R}} \sum_{j \in \mathcal{N}_i^r} \frac{1}{c_{i,r}} W_r^{(l)} h_j^{(l)} \right)$$

To prevent parameter explosion when relations $|\mathcal{R}|$ is large, RGCN uses **Basis Decomposition**:
$$W_r = \sum_{b=1}^{B} a_{rb} V_b$$
Where $V_b$ are shared base matrices, and $a_{rb}$ are relation-specific coefficients.

---

### 2. Scratch Python Implementation: Heterogeneous RGCN Layer

In [ ]:
import numpy as np

class RelationalGCNLayerScratch:
    def __init__(self, in_dim, out_dim, num_relations=3, num_bases=2):
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_relations = num_relations
        self.num_bases = num_bases
        
        # Self-loop transformation
        np.random.seed(42)
        self.W_root = np.random.normal(0, 0.1, (in_dim, out_dim))
        
        # Basis decomposition: shared basis matrices V_b
        self.bases = np.random.normal(0, 0.1, (num_bases, in_dim, out_dim))
        # Relation-specific linear coefficients a_{rb}
        self.coefficients = np.random.normal(0, 0.1, (num_relations, num_bases))
        
    def _get_relation_weight(self, r_idx):
        # Linear combination of shared basis matrices
        W_r = np.zeros((self.in_dim, self.out_dim))
        for b in range(self.num_bases):
            W_r += self.coefficients[r_idx, b] * self.bases[b]
        return W_r
        
    def forward(self, node_features, adjacency_by_relation):
        """
        node_features: (num_nodes, in_dim)
        adjacency_by_relation: dict {r: adj_matrix (num_nodes, num_nodes)}
        """
        num_nodes = len(node_features)
        out = np.dot(node_features, self.W_root) # Self connection
        
        # Aggregate messages across each relation type
        for r_idx, adj in adjacency_by_relation.items():
            W_r = self._get_relation_weight(r_idx)
            # Relation-specific degree normalization
            deg = np.sum(adj, axis=1, keepdims=True)
            deg[deg == 0] = 1.0 # Avoid division by zero
            norm_adj = adj / deg
            
            # Message Passing: A @ X @ W_r
            msg = np.dot(norm_adj, np.dot(node_features, W_r))
            out += msg
            
        return np.maximum(0, out) # ReLU activation

# Build Small Heterogeneous Graph (4 Nodes: 2 Authors, 2 Papers)
# Relations: 0: 'writes', 1: 'cites'
features = np.random.randn(4, 8)

adj_writes = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [0, 0, 0, 0],
    [0, 0, 0, 0]
])

adj_cites = np.array([
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 0, 0]
])

rgcn = RelationalGCNLayerScratch(in_dim=8, out_dim=4, num_relations=2, num_bases=2)
out_embeddings = rgcn.forward(features, {0: adj_writes, 1: adj_cites})

print("--- Relational GCN Message Passing ---")
print(f"Input Node Features Shape:  {features.shape}")
print(f"Output Node Embeddings:     {out_embeddings.shape}")
print("Heterogeneous edge relationships successfully aggregated into unified embeddings!")

### 3. Step-by-Step Code Walkthrough
1. **Basis Decomposition**: Shrinks weight parameters from $|\mathcal{R}| \times d_{\text{in}} \times d_{\text{out}}$ to $B \times d_{\text{in}} \times d_{\text{out}} + |\mathcal{R}| \times B$.
2. **Relational Normalization**: Divides by relation-specific degrees $c_{i,r} = |\mathcal{N}_i^r|$.
3. **Multi-Relational Aggregation**: Sums neighbor signals across edge categories.

---

### 4. Industry Applications & Trade-offs
- **Fraud Detection & Anti-Money Laundering:** Modeling diverse interaction types (account transfers, device sharing, IP sharing).
- **Trade-off:** Metapath extraction and multi-relational adjacency tensor storage requires substantial memory.

---

### 5. Top Interview Questions & Answers
**Q1: Why does standard GCN fail on multi-relational heterogeneous knowledge graphs?**
> **Answer:** Standard GCN applies a single shared weight matrix across all edges, losing semantic distinctions between different relationship types (e.g. treating "author writes paper" identically to "paper cites paper").

**Q2: How does Basis Decomposition prevent overfitting in RGCNs?**
> **Answer:** It represents each relation matrix as a linear combination of a small number of shared basis matrices, enforcing parameter sharing and preventing parameter explosion.